# Deep Cuisine Transfer — model

Trains the disentanglement seq2seq autoencoder (GRU encoder/decoder, style/content latent split, multi-task + adversarial classifiers in both directions), then does style transfer and evaluation.

**Run `01_preprocessing.ipynb` first.** It produces the artifacts this notebook loads:
- `data/recipes_train.csv`, `data/recipes_val.csv`, `data/recipes_test.csv`, `data/recipes_final.csv`
- `data/models/word2vec.wordvectors`
- `data/models/bow_vectorizer.joblib`

Training is resumable: rerunning the training cell picks up from `data/models/checkpoint_best_recon.pt` instead of starting over from random init.

In [ ]:
import ast
import os

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import KeyedVectors
from torch.utils.data import Dataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

def load_split(name):
    df = pd.read_csv(f'./data/recipes_{name}.csv', index_col=0)
    df['steps_tokens'] = df['steps_tokens'].apply(ast.literal_eval)
    df['steps_tokens_bow'] = df['steps_tokens_bow'].apply(ast.literal_eval)
    return df

train_df = load_split('train')
val_df = load_split('val')
test_df = load_split('test')

# name/steps (raw, human-readable text) for the style-transfer demo later - train/val/test
# splits only keep the tokenized columns, so this is joined back in by index when needed
df_final = pd.read_csv('./data/recipes_final.csv', index_col=0)[['name', 'steps']]

wv = KeyedVectors.load('./data/models/word2vec.wordvectors')

# the saved CountVectorizer was fit with analyzer=identity_analyzer (preprocessing.ipynb) -
# joblib/pickle only stores a reference to that function by name, so it must exist under this
# exact name here too or unpickling fails with AttributeError
def identity_analyzer(tokens):
    return tokens

vectorizer = joblib.load('./data/models/bow_vectorizer.joblib')

X_train_bow = vectorizer.transform(train_df['steps_tokens_bow'])
X_val_bow = vectorizer.transform(val_df['steps_tokens_bow'])
X_test_bow = vectorizer.transform(test_df['steps_tokens_bow'])

print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
seq_len = int(train_df['steps_tokens'].apply(len).quantile(0.95))

def tokens_to_matrix(tokens, wv, seq_len):
    matrix = np.zeros((seq_len, wv.vector_size), dtype=np.float32)
    for i, token in enumerate(tokens[:seq_len]):
        if token in wv:
            matrix[i] = wv[token]
    return matrix

## Vocabulary and target indices (for CrossEntropyLoss)

In [ ]:
PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN = '<PAD>', '<UNK>', '<SOS>', '<EOS>'

word2idx = dict(wv.key_to_index)
word2idx[PAD_TOKEN] = len(word2idx)
PAD_IDX = word2idx[PAD_TOKEN]
word2idx[UNK_TOKEN] = len(word2idx)
UNK_IDX = word2idx[UNK_TOKEN]
word2idx[SOS_TOKEN] = len(word2idx)
SOS_IDX = word2idx[SOS_TOKEN]
word2idx[EOS_TOKEN] = len(word2idx)
EOS_IDX = word2idx[EOS_TOKEN]

VOCAB_SIZE = len(word2idx)

def tokens_to_indices(tokens, word2idx, seq_len):
    # leave room for EOS, then pad - EOS is included in the loss (unlike PAD),
    # giving the model a signal for where to stop generating
    tokens = tokens[:seq_len - 1]
    indices = [word2idx.get(t, UNK_IDX) for t in tokens]
    indices.append(EOS_IDX)
    indices += [PAD_IDX] * (seq_len - len(indices))
    return indices

## GRU Encoder

In [ ]:
class GRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, style_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.style_dim = style_dim
        self.content_dim = hidden_dim - style_dim
        self.style_head = nn.Linear(hidden_dim, style_dim)
        self.content_head = nn.Linear(hidden_dim, self.content_dim)

    def forward(self, x, lengths):
        # pack so the GRU stops at each sequence's real last token instead of
        # continuing to update hidden state through trailing zero-padding
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        hidden = hidden.squeeze(0)                    # (batch, hidden_dim)
        style_latent = self.style_head(hidden)          # (batch, style_dim)
        content_latent = self.content_head(hidden)       # (batch, content_dim)
        return style_latent, content_latent

# encoder turns each recipe into one hidden vector, split via two heads into
# style_latent (e.g. cuisine) and content_latent (recipe content)
input_dim = wv.vector_size
hidden_dim = 512
STYLE_DIM = 64
encoder = GRUEncoder(input_dim, hidden_dim, STYLE_DIM).to(device)

## GRU Decoder

In [ ]:
vocab_size = VOCAB_SIZE
MAX_LEN = seq_len

def block_repeat_ngrams(logits_t, generated, ngram_size):
    # standard no-repeat-ngram decoding constraint: for each sequence in the batch, find
    # tokens that would complete an n-gram already seen earlier in that sequence, and ban them
    for b in range(logits_t.size(0)):
        seq = generated[b]
        if seq:
            # separate from the n-gram check below: that only catches a token repeating a
            # transition seen earlier in this exact sequence, so an immediate stutter like
            # "the the" slips through the first time it happens (nothing to compare it to yet)
            logits_t[b, seq[-1]] = float('-inf')
        if len(seq) < ngram_size - 1:
            continue
        prefix = tuple(seq[-(ngram_size - 1):])
        banned = {
            seq[i + ngram_size - 1]
            for i in range(len(seq) - ngram_size + 1)
            if tuple(seq[i:i + ngram_size - 1]) == prefix
        }
        if banned:
            logits_t[b, list(banned)] = float('-inf')
    return logits_t

class GRUDecoder(nn.Module):
    def __init__(self, hidden_dim, vocab_size, embed_dim, sos_idx, pad_idx, unk_idx, max_len):
        super().__init__()
        self.max_len = max_len
        self.sos_idx = sos_idx
        self.unk_idx = unk_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru_cell = nn.GRUCell(embed_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, hidden, target=None, teacher_forcing_ratio=0.5, no_repeat_ngram_size=0):
        # hidden: (batch, hidden_dim) - encoder context vector, used as initial GRU state
        batch_size = hidden.size(0)
        device = hidden.device
        max_len = target.size(1) if target is not None else self.max_len

        input_idx = torch.full((batch_size,), self.sos_idx, dtype=torch.long, device=device)
        h = hidden
        outputs = []
        generated = [[] for _ in range(batch_size)] if no_repeat_ngram_size > 0 else None

        for t in range(max_len):
            embedded = self.embedding(input_idx)   # (batch, embed_dim)
            h = self.gru_cell(embedded, h)          # (batch, hidden_dim)
            logits_t = self.fc_out(h)               # (batch, vocab_size)

            use_teacher_forcing = target is not None and torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                outputs.append(logits_t.unsqueeze(1))
                input_idx = target[:, t]
            else:
                if target is None:
                    # pure free-running generation - as opposed to a training step that simply
                    # wasn't teacher-forced this time (target is still the ground truth there,
                    # and masking <UNK> would zero out the loss gradient whenever the true next
                    # token actually is <UNK>). <UNK> is a training-target artifact for rare
                    # words, never a word we want to see in real generated output.
                    logits_t[:, self.unk_idx] = float('-inf')
                    if no_repeat_ngram_size > 0:
                        logits_t = block_repeat_ngrams(logits_t, generated, no_repeat_ngram_size)
                outputs.append(logits_t.unsqueeze(1))
                input_idx = logits_t.argmax(dim=-1)
                if no_repeat_ngram_size > 0:
                    for b in range(batch_size):
                        generated[b].append(input_idx[b].item())

        return torch.cat(outputs, dim=1)  # (batch, max_len, vocab_size)

decoder = GRUDecoder(hidden_dim, vocab_size, input_dim, SOS_IDX, PAD_IDX, UNK_IDX, MAX_LEN).to(device)

## Seq2Seq Autoencoder — training (input recipe = target recipe)

In [ ]:
CUISINE2IDX = {'Italian': 0, 'Indian': 1}
BOW_VOCAB_SIZE = X_train_bow.shape[1]

class RecipeDataset(Dataset):
    def __init__(self, df, wv, word2idx, seq_len, cuisine2idx, bow_matrix):
        self.tokens = df['steps_tokens'].tolist()
        self.cuisines = df['cuisine'].tolist()
        self.wv = wv
        self.word2idx = word2idx
        self.seq_len = seq_len
        self.cuisine2idx = cuisine2idx
        self.bow_matrix = bow_matrix

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        tokens = self.tokens[idx]
        x = tokens_to_matrix(tokens, self.wv, self.seq_len)
        y = tokens_to_indices(tokens, self.word2idx, self.seq_len)
        length = max(1, min(len(tokens), self.seq_len))  # real (non-padded) token count fed to the encoder
        style_label = self.cuisine2idx[self.cuisines[idx]]
        bow_target = (self.bow_matrix[idx].toarray().ravel() > 0).astype(np.float32)  # multi-label word presence
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(style_label, dtype=torch.long),
            torch.tensor(bow_target, dtype=torch.float32),
        )

BATCH_SIZE = 32

train_dataset = RecipeDataset(train_df, wv, word2idx, seq_len, CUISINE2IDX, X_train_bow)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

class Seq2SeqAutoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, lengths, target=None, teacher_forcing_ratio=0.5, no_repeat_ngram_size=0, return_latents=False):
        style_latent, content_latent = self.encoder(x, lengths)
        context = torch.cat([style_latent, content_latent], dim=-1)  # merge back for the decoder
        logits = self.decoder(
            context, target=target, teacher_forcing_ratio=teacher_forcing_ratio,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )
        if return_latents:
            return logits, style_latent, content_latent
        return logits

recon_criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
style_criterion = nn.CrossEntropyLoss()
content_criterion = nn.BCEWithLogitsLoss()  # multi-label: each word independently yes/no
adv_style_criterion = nn.CrossEntropyLoss()
adv_content_criterion = nn.BCEWithLogitsLoss()

def categorical_entropy(logits):
    log_probs = torch.log_softmax(logits, dim=-1)
    return -(log_probs.exp() * log_probs).sum(dim=-1).mean()

def bernoulli_entropy(logits):
    # per-label entropy of an independent Bernoulli, computed via logsigmoid for stability.
    # mean (not sum) over labels so this stays on the same ~ln(2) scale as categorical_entropy
    # regardless of BOW_VOCAB_SIZE - summing over thousands of labels would dwarf every other loss term.
    log_p = F.logsigmoid(logits)
    log_1m_p = F.logsigmoid(-logits)
    p = log_p.exp()
    return -(p * log_p + (1 - p) * log_1m_p).mean(dim=-1).mean()

CHECKPOINT_PATH = './data/models/checkpoint.pt'
BEST_RECON_CHECKPOINT_PATH = './data/models/checkpoint_best_recon.pt'
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def make_checkpoint(epoch, best_recon_loss):
    return {
        'epoch': epoch,
        'best_recon_loss': best_recon_loss,
        'model': model.state_dict(),
        'style_classifier': style_classifier.state_dict(),
        'content_classifier': content_classifier.state_dict(),
        'adv_style_classifier': adv_style_classifier.state_dict(),
        'adv_content_classifier': adv_content_classifier.state_dict(),
        'hparams': {
            'input_dim': input_dim,
            'hidden_dim': hidden_dim,
            'style_dim': STYLE_DIM,
            'vocab_size': VOCAB_SIZE,
            'bow_vocab_size': BOW_VOCAB_SIZE,
            'seq_len': seq_len,
            'pad_idx': PAD_IDX,
            'sos_idx': SOS_IDX,
            'unk_idx': UNK_IDX,
        },
    }

# only build the model/classifiers/optimizers the first time this cell runs, so rerunning
# the cell resumes training instead of starting over from random init
if 'model' not in globals():
    model = Seq2SeqAutoencoder(encoder, decoder).to(device)
    style_classifier = nn.Linear(STYLE_DIM, len(CUISINE2IDX)).to(device)

    # content classifier: content_vector -> predicted BoW word presence
    content_classifier = nn.Sequential(
        nn.Linear(encoder.content_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, BOW_VOCAB_SIZE),
    ).to(device)

    # adversarial style classifier (J_adv(s) in the paper): content_latent should carry no style
    # info, so the encoder is trained to maximize this classifier's prediction entropy
    adv_style_classifier = nn.Sequential(
        nn.Linear(encoder.content_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, len(CUISINE2IDX)),
    ).to(device)

    # adversarial content classifier (J_adv(c) in the paper): style_latent should carry no content
    # info, so the encoder is trained to maximize this classifier's per-word prediction entropy
    adv_content_classifier = nn.Sequential(
        nn.Linear(encoder.style_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, BOW_VOCAB_SIZE),
    ).to(device)

    # paper: Adam for the autoencoder + multi-task heads, RMSProp for the adversarial discriminators
    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(style_classifier.parameters()) + list(content_classifier.parameters()),
        lr=1e-3,
    )
    adv_style_optimizer = torch.optim.RMSprop(adv_style_classifier.parameters(), lr=1e-3)
    adv_content_optimizer = torch.optim.RMSprop(adv_content_classifier.parameters(), lr=1e-3)

    # resume from the best checkpoint on disk (e.g. after a kernel restart or an interrupted
    # run) instead of starting from random init
    best_recon_loss = float('inf')
    loss_history = []  # per-epoch metrics across all runs of this cell, for the loss-curve plot below
    if os.path.exists(BEST_RECON_CHECKPOINT_PATH):
        checkpoint = torch.load(BEST_RECON_CHECKPOINT_PATH, map_location=device, weights_only=False)
        if 'model' in checkpoint:
            model.load_state_dict(checkpoint['model'])
            style_classifier.load_state_dict(checkpoint['style_classifier'])
            content_classifier.load_state_dict(checkpoint['content_classifier'])
            adv_style_classifier.load_state_dict(checkpoint['adv_style_classifier'])
            adv_content_classifier.load_state_dict(checkpoint['adv_content_classifier'])
            best_recon_loss = checkpoint['best_recon_loss']
            print(f"Resumed from {BEST_RECON_CHECKPOINT_PATH}, recon loss {best_recon_loss:.4f}")
        else:
            # older checkpoint format: a bare encoder/decoder state_dict, no classifiers or
            # metadata - still a valid warm start since GRUEncoder/GRUDecoder are unchanged
            model.load_state_dict(checkpoint)
            print(f"Resumed encoder/decoder only from older-format {BEST_RECON_CHECKPOINT_PATH} (classifiers start fresh)")
    else:
        print(f"No checkpoint found at {BEST_RECON_CHECKPOINT_PATH}, training from scratch")

# rerun this cell as many times as you like - it resumes from the best checkpoint each time
NUM_EPOCHS = 20
TEACHER_FORCING_RATIO = 0.5
STYLE_LOSS_WEIGHT = 1.0
CONTENT_LOSS_WEIGHT = 1.0
ADV_STYLE_LOSS_WEIGHT = 2.0
ADV_CONTENT_LOSS_WEIGHT = 2.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    style_classifier.train()
    content_classifier.train()
    adv_style_classifier.train()
    adv_content_classifier.train()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_style_loss = 0.0
    total_content_loss = 0.0
    total_adv_style_encoder_loss = 0.0
    total_adv_content_encoder_loss = 0.0
    total_adv_style_clf_loss = 0.0
    total_adv_content_clf_loss = 0.0
    correct_adv_style = 0
    total_adv = 0

    for batch_x, batch_y, batch_len, batch_style, batch_bow in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_style = batch_style.to(device)
        batch_bow = batch_bow.to(device)
        optimizer.zero_grad()

        logits, style_latent, content_latent = model(
            batch_x, batch_len, target=batch_y, teacher_forcing_ratio=TEACHER_FORCING_RATIO, return_latents=True
        )
        recon_loss = recon_criterion(logits.permute(0, 2, 1), batch_y)   # (batch, vocab_size, seq_len) vs (batch, seq_len)

        style_logits = style_classifier(style_latent)
        style_loss = style_criterion(style_logits, batch_style)

        content_logits = content_classifier(content_latent)
        content_loss = content_criterion(content_logits, batch_bow)

        # encoder wants both adversarial classifiers to be as uninformative (max-entropy) as possible
        adv_style_logits_for_encoder = adv_style_classifier(content_latent)
        adv_style_encoder_loss = -categorical_entropy(adv_style_logits_for_encoder)

        adv_content_logits_for_encoder = adv_content_classifier(style_latent)
        adv_content_encoder_loss = -bernoulli_entropy(adv_content_logits_for_encoder)

        loss = (
            recon_loss
            + STYLE_LOSS_WEIGHT * style_loss
            + CONTENT_LOSS_WEIGHT * content_loss
            + ADV_STYLE_LOSS_WEIGHT * adv_style_encoder_loss
            + ADV_CONTENT_LOSS_WEIGHT * adv_content_encoder_loss
        )
        loss.backward()
        optimizer.step()

        # adversarial classifiers are trained separately, on detached latents, to actually
        # get as good as possible at their task (the encoder above is fighting this)
        adv_style_optimizer.zero_grad()
        adv_style_clf_logits = adv_style_classifier(content_latent.detach())
        adv_style_clf_loss = adv_style_criterion(adv_style_clf_logits, batch_style)
        adv_style_clf_loss.backward()
        adv_style_optimizer.step()

        adv_content_optimizer.zero_grad()
        adv_content_clf_logits = adv_content_classifier(style_latent.detach())
        adv_content_clf_loss = adv_content_criterion(adv_content_clf_logits, batch_bow)
        adv_content_clf_loss.backward()
        adv_content_optimizer.step()

        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_style_loss += style_loss.item()
        total_content_loss += content_loss.item()
        total_adv_style_encoder_loss += adv_style_encoder_loss.item()
        total_adv_content_encoder_loss += adv_content_encoder_loss.item()
        total_adv_style_clf_loss += adv_style_clf_loss.item()
        total_adv_content_clf_loss += adv_content_clf_loss.item()
        correct_adv_style += (adv_style_clf_logits.argmax(dim=-1) == batch_style).sum().item()
        total_adv += batch_style.size(0)

    n_batches = len(train_loader)
    avg_recon_loss = total_recon_loss / n_batches
    avg_adv_style_clf_acc = correct_adv_style / total_adv
    print(
        f'epoch {epoch}/{NUM_EPOCHS} '
        f'| loss {total_loss / n_batches:.4f} '
        f'| recon {avg_recon_loss:.4f} | style {total_style_loss / n_batches:.4f} '
        f'| content {total_content_loss / n_batches:.4f} '
        f'| adv_style_enc {total_adv_style_encoder_loss / n_batches:.4f} | adv_style_clf {total_adv_style_clf_loss / n_batches:.4f} '
        f'| adv_content_enc {total_adv_content_encoder_loss / n_batches:.4f} | adv_content_clf {total_adv_content_clf_loss / n_batches:.4f} '
        f'| adv_style_clf_acc {avg_adv_style_clf_acc:.4f}'
    )

    # keyed by cumulative epoch count (len(loss_history) + 1), not the per-run `epoch` variable,
    # so the loss-curve plot below stays continuous across repeated runs of this cell
    loss_history.append({
        'epoch': len(loss_history) + 1,
        'recon_loss': avg_recon_loss,
        'adv_style_clf_acc': avg_adv_style_clf_acc,
    })

    # recon_loss doesn't monotonically improve (style/adv losses can pull it back up), so
    # the final epoch isn't necessarily the best - checkpoint whenever it improves instead
    if avg_recon_loss < best_recon_loss:
        best_recon_loss = avg_recon_loss
        torch.save(make_checkpoint(epoch, best_recon_loss), BEST_RECON_CHECKPOINT_PATH)
        print(f'  -> new best recon loss {best_recon_loss:.4f}, saved to {BEST_RECON_CHECKPOINT_PATH}')

torch.save(make_checkpoint(NUM_EPOCHS, best_recon_loss), CHECKPOINT_PATH)

# everything downstream (style vectors, style transfer generation, held-out eval) should run
# on the best checkpoint seen so far, not whichever epoch this run happened to end on
best_checkpoint = torch.load(BEST_RECON_CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model'])
style_classifier.load_state_dict(best_checkpoint['style_classifier'])
content_classifier.load_state_dict(best_checkpoint['content_classifier'])
adv_style_classifier.load_state_dict(best_checkpoint['adv_style_classifier'])
adv_content_classifier.load_state_dict(best_checkpoint['adv_content_classifier'])

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(loss_history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df['epoch'], history_df['recon_loss'])
axes[0].set_title('Reconstruction loss')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('recon loss')

axes[1].plot(history_df['epoch'], history_df['adv_style_clf_acc'])
axes[1].axhline(0.5, color='red', linestyle='--', label='chance level')
axes[1].set_title('Adversarial style classifier accuracy\n(content latent -> cuisine)')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
idx2word = {idx: word for word, idx in word2idx.items()}
NO_REPEAT_NGRAM_SIZE = 3  # blocks repeated trigrams during free-running generation (greedy decoding loops otherwise)

model.eval()
with torch.no_grad():
    sample_x, sample_y, sample_len, sample_style, sample_bow = next(iter(train_loader))
    sample_x = sample_x.to(device)
    logits = model(
        sample_x, sample_len, target=None, teacher_forcing_ratio=0.0,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )  # free-running generation, no teacher forcing
    predicted_indices = logits.argmax(dim=-1)  # (batch, seq_len)

def indices_to_tokens(indices):
    # stop at the first EOS - anything after it is not part of the generated content
    tokens = []
    for i in indices:
        if i == EOS_IDX:
            break
        if i != PAD_IDX:
            tokens.append(idx2word[i])
    return tokens

recipe_idx = 0
original_tokens = indices_to_tokens(sample_y[recipe_idx].tolist())
predicted_tokens = indices_to_tokens(predicted_indices[recipe_idx].tolist())

print('ORIGINAL: ', ' '.join(original_tokens))
print('PREDICTED:', ' '.join(predicted_tokens))

## Held-out test set evaluation

In [ ]:
test_dataset = RecipeDataset(test_df, wv, word2idx, seq_len, CUISINE2IDX, X_test_bow)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model.eval()
style_classifier.eval()
adv_style_classifier.eval()
total_recon, correct_style, correct_adv_style, n_batches, n_rows = 0.0, 0, 0, 0, 0
with torch.no_grad():
    for batch_x, batch_y, batch_len, batch_style, batch_bow in test_loader:
        batch_x, batch_y, batch_style = batch_x.to(device), batch_y.to(device), batch_style.to(device)
        logits, style_latent, content_latent = model(
            batch_x, batch_len, target=batch_y, teacher_forcing_ratio=0.0, return_latents=True
        )
        total_recon += recon_criterion(logits.permute(0, 2, 1), batch_y).item()
        correct_style += (style_classifier(style_latent).argmax(dim=-1) == batch_style).sum().item()
        correct_adv_style += (adv_style_classifier(content_latent).argmax(dim=-1) == batch_style).sum().item()
        n_batches += 1
        n_rows += batch_style.size(0)

print(f'test recon loss: {total_recon / n_batches:.4f}')
print(f'test style_clf accuracy (style latent -> cuisine): {correct_style / n_rows:.4f}')
print(f'test adv_style_clf accuracy (content latent -> cuisine, chance = 0.5): {correct_adv_style / n_rows:.4f}')

## Style transfer

Average the style vectors of every training example within a cuisine class to get an empirical `Enc(S_T)` for that class (as in the paper's section 5.2). To transfer a recipe, encode it to get its content vector, discard its own style vector, and decode using the *target* cuisine's average style vector instead.

In [ ]:
@torch.no_grad()
def compute_class_style_vectors(model, dataset, cuisine2idx, device, batch_size=128):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    style_sums = {idx: torch.zeros(model.encoder.style_dim, device=device) for idx in cuisine2idx.values()}
    style_counts = {idx: 0 for idx in cuisine2idx.values()}

    model.eval()
    for batch_x, _, batch_len, batch_style, _ in loader:
        batch_x = batch_x.to(device)
        batch_style = batch_style.to(device)
        style_latent, _ = model.encoder(batch_x, batch_len)
        for cls_idx in style_counts:
            mask = batch_style == cls_idx
            if mask.any():
                style_sums[cls_idx] += style_latent[mask].sum(dim=0)
                style_counts[cls_idx] += mask.sum().item()

    return {idx: style_sums[idx] / style_counts[idx] for idx in style_counts}

IDX2CUISINE = {idx: name for name, idx in CUISINE2IDX.items()}
class_style_vectors = compute_class_style_vectors(model, train_dataset, CUISINE2IDX, device)
for idx, vec in class_style_vectors.items():
    print(f"{IDX2CUISINE[idx]}: style vector norm = {vec.norm().item():.4f}")

# persist so a later session (or a separate generation script) doesn't need to re-run the
# whole training set through the encoder just to get these
for idx, vec in class_style_vectors.items():
    torch.save(vec, f'./data/models/style_avg_{IDX2CUISINE[idx].lower()}.pt')

In [ ]:
@torch.no_grad()
def transfer_style(model, tokens, wv, seq_len, target_style_vector, device):
    model.eval()
    x = tokens_to_matrix(tokens, wv, seq_len)
    x = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
    length = torch.tensor([max(1, min(len(tokens), seq_len))], dtype=torch.long)

    _, content_latent = model.encoder(x, length)
    context = torch.cat([target_style_vector.unsqueeze(0), content_latent], dim=-1)
    logits = model.decoder(
        context, target=None, teacher_forcing_ratio=0.0, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )  # free-running generation
    predicted_indices = logits.argmax(dim=-1).squeeze(0).tolist()
    return indices_to_tokens(predicted_indices)

OPPOSITE_CUISINE = {'Italian': 'Indian', 'Indian': 'Italian'}
N_PER_CUISINE = 6

# sample from val_df (held out, never seen during training) rather than train_df, and join
# back the raw name/steps text (dropped from the tokenized train/val/test splits) for a
# human-readable results table
demo_samples = pd.concat([
    g.sample(n=min(N_PER_CUISINE, len(g)), random_state=42)
    for _, g in val_df.groupby('cuisine')
]).join(df_final)
demo_samples['target_cuisine'] = demo_samples['cuisine'].map(OPPOSITE_CUISINE)

generated_texts = [
    ' '.join(transfer_style(model, row['steps_tokens'], wv, seq_len, class_style_vectors[CUISINE2IDX[row['target_cuisine']]], device))
    for _, row in demo_samples.iterrows()
]

results = pd.DataFrame({
    'recipe_name': demo_samples['name'].tolist(),
    'original_cuisine': demo_samples['cuisine'].tolist(),
    'original_text': demo_samples['steps'].tolist(),
    'generated_text': generated_texts,
    'target_cuisine': demo_samples['target_cuisine'].tolist(),
})
results.to_csv('./data/style_transfer_results.csv')

for _, row in results.head(4).iterrows():
    print(f"\n{row['recipe_name']} ({row['original_cuisine']} -> {row['target_cuisine']})")
    print('ORIGINAL:   ', row['original_text'])
    print('TRANSFERRED:', row['generated_text'])

results

## Independent evaluation classifier

A separate classifier, trained only on the *original* (non-transferred) recipe text using averaged Word2Vec features. It never touches the autoencoder's style/content latents, so using it to score the style-transferred text afterwards isn't circular - it mirrors the paper's section 6 evaluation (84.5% accuracy on real text as a baseline, then the same classifier is used to measure how convincingly the transferred text reads as the target cuisine).

In [ ]:
def recipe_to_avg_vector(tokens, wv):
    vectors = [wv[t] for t in tokens if t in wv]
    if not vectors:
        return np.zeros(wv.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

def build_eval_features(df, wv):
    X = np.stack([recipe_to_avg_vector(tokens, wv) for tokens in df['steps_tokens']])
    y = np.array([CUISINE2IDX[c] for c in df['cuisine']], dtype=np.int64)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

X_eval_train, y_eval_train = build_eval_features(train_df, wv)
X_eval_val, y_eval_val = build_eval_features(val_df, wv)
X_eval_train, y_eval_train = X_eval_train.to(device), y_eval_train.to(device)
X_eval_val, y_eval_val = X_eval_val.to(device), y_eval_val.to(device)

class StyleEvalClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, len(CUISINE2IDX)),
        )

    def forward(self, x):
        return self.net(x)

eval_classifier = StyleEvalClassifier(wv.vector_size).to(device)
eval_optimizer = torch.optim.Adam(eval_classifier.parameters(), lr=1e-3)
eval_criterion = nn.CrossEntropyLoss()

EVAL_EPOCHS = 30
EVAL_BATCH_SIZE = 64

for epoch in range(1, EVAL_EPOCHS + 1):
    eval_classifier.train()
    perm = torch.randperm(X_eval_train.size(0))
    total_loss = 0.0
    for start in range(0, X_eval_train.size(0), EVAL_BATCH_SIZE):
        idx = perm[start:start + EVAL_BATCH_SIZE]
        batch_x, batch_y = X_eval_train[idx], y_eval_train[idx]

        eval_optimizer.zero_grad()
        logits = eval_classifier(batch_x)
        loss = eval_criterion(logits, batch_y)
        loss.backward()
        eval_optimizer.step()
        total_loss += loss.item() * batch_x.size(0)

    if epoch % 5 == 0 or epoch == EVAL_EPOCHS:
        eval_classifier.eval()
        with torch.no_grad():
            val_acc = (eval_classifier(X_eval_val).argmax(dim=-1) == y_eval_val).float().mean().item()
        print(f"epoch {epoch:3d} | train_loss {total_loss / X_eval_train.size(0):.4f} | val_acc {val_acc:.4f}")

### Transfer strength

For held-out recipes (val_df, never seen by the autoencoder), transfer each to the *opposite* cuisine and check whether the independent classifier - trained only on real text - agrees that the result reads as the target style. This is the paper's transfer-strength metric (73% in the paper, vs. 84.5% on real text).

In [ ]:
@torch.no_grad()
def score_transfer_strength(model, eval_classifier, df, wv, seq_len, class_style_vectors, cuisine2idx, device, sample_size=200):
    sample_df = df.sample(n=min(sample_size, len(df)), random_state=42)

    correct = 0
    total = 0
    for _, row in sample_df.iterrows():
        source_idx = cuisine2idx[row['cuisine']]
        target_idx = 1 - source_idx
        transferred_tokens = transfer_style(
            model, row['steps_tokens'], wv, seq_len, class_style_vectors[target_idx], device
        )
        feature = torch.tensor(recipe_to_avg_vector(transferred_tokens, wv), dtype=torch.float32, device=device).unsqueeze(0)
        predicted_idx = eval_classifier(feature).argmax(dim=-1).item()
        correct += int(predicted_idx == target_idx)
        total += 1

    return correct / total

transfer_strength = score_transfer_strength(
    model, eval_classifier, val_df, wv, seq_len, class_style_vectors, CUISINE2IDX, device
)
print(f"Transfer strength (independent classifier agrees with target style): {transfer_strength:.4f}")